In [ ]:
%sql

--Cria catalog e schema
CREATE CATALOG IF NOT EXISTS lh_nautical;
CREATE SCHEMA IF NOT EXISTS lh_nautical.bronze;

In [ ]:
# Isolando a infraestrutura de conexão
from databricks.sdk.runtime import dbutils, spark 

db_user = dbutils.secrets.get(scope="sqlserver", key="username")
db_password = dbutils.secrets.get(scope="sqlserver", key="password")
db_host = dbutils.secrets.get(scope="sqlserver", key="host")
db_port = dbutils.secrets.get(scope="sqlserver", key="port")
db_database = dbutils.secrets.get(scope="sqlserver", key="db")

jdbc_url = f"jdbc:sqlserver://{db_host}:{db_port};databaseName={db_database}"

In [ ]:
import json
from pyspark.sql import functions as F

# 1. Carrega dinamicamente o mapeamento com as 22 tabelas que validamos
with open('tabelas_config.json', 'r', encoding='utf-8') as f:
    mapeamento_tabelas = json.load(f)

# 2. Mapeamento de Políticas de Governança (LGPD / PII)
politica_governanca = {
    "customers": ["legal_name", "trade_name", "tax_id", "email"],
    "employees": ["full_name", "cpf", "email"],
    "suppliers": ["legal_name", "trade_name", "tax_id"],
    "fiscal_invoices": ["nfe_access_key", "xml_storage_uri"],
    "goods_receipts": ["notes"]
}

print(f"Iniciando loop de ingestão com Governança Ativa para {len(mapeamento_tabelas)} tabelas.")

# 3. Loop de extração e mascaramento automatizado
for tabela in mapeamento_tabelas.keys():
    print(f"\n[EXTRAÇÃO] Buscando dados da tabela: {tabela}...")
    
    jdbc_options = {
        "url":                    jdbc_url,
        "dbtable":                tabela,
        "user":                   db_user,
        "password":               db_password,
        "driver":                 "com.microsoft.sqlserver.jdbc.SQLServerDriver",
        "encrypt":                "true",
        "trustServerCertificate": "true",
        "loginTimeout":           "30",
        "fetchsize":              "1000",
    }
    
    # Faz a leitura do banco de origem (SQL Server)
    df_temp = (
        spark.read 
        .format("jdbc") 
        .options(**jdbc_options) 
        .load()
    )
    
    # Aplica o Hashing se a tabela possuir regras de governança
    if tabela in politica_governanca:
        for coluna_sensivel in politica_governanca[tabela]:
            if coluna_sensivel in df_temp.columns:
                print(f"  └─► [GOVERNANÇA] Aplicando SHA-256 na coluna sensível: '{coluna_sensivel}'")
                df_temp = df_temp.withColumn(
                    coluna_sensivel, 
                    F.sha2(F.col(coluna_sensivel).cast("string"), 256)
                )
    
    # Grava o dado mascarado direto na tabela gerenciada do Unity Catalog
    output_table = f"lh_nautical.bronze.{tabela}"
    df_temp.write.format("delta").mode("overwrite").saveAsTable(output_table)
    print(f"[SUCESSO] Tabela {output_table} salva com segurança!")